In [30]:
import ipydatetime
import datetime
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
import urllib.request
import json
from __future__ import print_function
from ipywidgets import interact, interactive, fixed, interact_manual, Layout, Label, HBox
import ipywidgets as ipw
from IPython.display import display, clear_output
import html
import pandas as pd
import requests
import folium
from functools import partial
import time
import numpy as np

In [36]:
display(ipw.HTML("<h1 style='margin-left:35%;margin-bottom:25px'>Fuel Delivery Truck Routing Optimization Tool</h1>"))
display(ipw.HTML("<h2 style='margin-left:25%;margin-bottom:25px'>Add Delivery Locations</h2>"))
key=ipw.Text(description="LocationIQ API Key",value='pk.8486eabe4a0e3eac50b996f802868e59',style={'description_width': 'initial'},layout=Layout(display='none'))
display(key)
locations=pd.DataFrame(columns=['Address','Longitude','Latitude','Diesel Delivery Amount','Dyed Diesel Delivery Amount','Delivery Window Start','Delivery Window End'])

def add_location(x):
    global capacity
    global capacity2
    global results
    location=[d for d in results if select.value in d['display_name']]
    global locations
    locations=locations.append({'Address':location[0]['display_name'],'Longitude':location[0]['lon'],'Latitude':location[0]['lat'],'Diesel Delivery Amount':capacity.value,'Dyed Diesel Delivery Amount':capacity2.value,'Delivery Window Start':windowstart.value,'Delivery Window End':windowend.value},ignore_index=True)
    return 

def get_coordinates(address):
    API_key=key.value
    if address!="":
          request = 'https://us1.locationiq.com/v1/search.php?'
          address= urllib.parse.quote(address)
          request = request + 'key=' + API_key + '&q=' + address + '&format=json'
          jsonResult = urllib.request.urlopen(request).read()
          global results 
          results= json.loads(jsonResult)
          global display_name
          display_name=[]
          for result in results:
                display_name.append(result['display_name'])
          global select
          select=ipw.Select(
            options=display_name,
            layout={'width': 'max-content'}, # If the items' names are long
            style={'description_width': 'initial'},
            description='Select correct address:',
            disabled=False
            )
          display(select)         
          return

def get_depot_coordinates(address):
    API_key=key.value
    if address!="":
          request = 'https://us1.locationiq.com/v1/search.php?'
          address= urllib.parse.quote(address)
          request = request + 'key=' + API_key + '&q=' + address + '&format=json'
          jsonResult = urllib.request.urlopen(request).read()
          global depot_results 
          depot_results= json.loads(jsonResult)
          global display_name
          display_name=[]
          for result in depot_results:
                display_name.append(result['display_name'])
          global select2
          select2=ipw.Select(
            options=display_name,
            layout={'width': 'max-content'}, # If the items' names are long
            style={'description_width': 'initial'},
            description='Select correct address:',
            disabled=False
            )
          display(select2)         
          return

my_interact_manual = interact_manual.options(manual_name="Search Address")
results=my_interact_manual(get_coordinates,address=ipw.Text(value="",placeholder='Type address',description="Location Address",layout=Layout(width='300px'),style={'description_width': 'initial'}),API_key=fixed(key))
run_add_location=ipw.Button(description='Add Location',button_style='primary')
run_add_location.on_click(add_location)

capacity=ipw.IntText(value=0,description="Diesel Delivery Amount:",style={'description_width': 'initial'})
capacity2=ipw.IntText(value=0,description="Dyed Diesel Delivery Amount:",style={'description_width': 'initial'})
windowstart=ipydatetime.TimePicker(description='Delivery Window Start',style={'description_width': 'initial'},value=datetime.time(7, 00))
windowend=ipydatetime.TimePicker(description='Delivery Window End',style={'description_width': 'initial'},value=datetime.time(19, 00))
box=HBox([capacity,capacity2,windowstart,windowend])
display(box)
output=ipw.Output()
def handle_submit(sender):
    with output:
        clear_output()
        display(ipw.HTML("<h2>Delivery Locations</h2>"))
        display(locations)  
run_add_location.on_click(handle_submit)



display(run_add_location)

display(output)

HTML(value="<h1 style='margin-left:35%;margin-bottom:25px'>Fuel Delivery Truck Routing Optimization Tool</h1>"…

HTML(value="<h2 style='margin-left:25%;margin-bottom:25px'>Add Delivery Locations</h2>")

Text(value='pk.8486eabe4a0e3eac50b996f802868e59', description='LocationIQ API Key', layout=Layout(display='non…

interactive(children=(Text(value='', description='Location Address', layout=Layout(width='300px'), placeholder…

Button(button_style='primary', description='Add Location', style=ButtonStyle())

Output()

In [20]:
vehicles=pd.DataFrame(columns=['Truck No.','Diesel Capacity','Dyed Diesel Capacity','Starting Diesel Capacity','Starting Dyed Diesel Capacity'])
def add_truck(x):
    global truck
    global vehicles
    number=len(vehicles)+1
    vehicles=vehicles.append({'Truck No.':number,'Diesel Capacity':diesel.value,'Dyed Diesel Capacity':diesel2.value,'Starting Diesel Capacity':startdiesel.value,'Starting Dyed Diesel Capacity':startdiesel2.value},ignore_index=True)
    return vehicles
diesel=ipw.IntText(description='Diesel Capacity',style={'description_width': 'initial'})
diesel2=ipw.IntText(description='Dyed Diesel Capacity',style={'description_width': 'initial'})
startdiesel=ipw.IntText(description='Starting Diesel Capacity',style={'description_width': 'initial'})
startdiesel2=ipw.IntText(description='Starting Dyed Diesel Capacity',style={'description_width': 'initial'})
add_vehicle=ipw.Button(description='Add Vehicle',button_style='primary')
add_vehicle.on_click(add_truck)
output3=ipw.Output()
def handle_submit3(sender):
    with output3:
        clear_output()
        display(ipw.HTML("<h2>Fuel Trucks</h2>"))
        display(vehicles)  
add_vehicle.on_click(handle_submit3)

display(ipw.HTML("<h2 style='margin-left:25%;margin-bottom:25px'>Add Delivery Trucks</h2>"))
box2=HBox([diesel,diesel2])
box3=HBox([startdiesel,startdiesel2])
display(box2,box3)
display(add_vehicle)
display(output3)

HTML(value="<h2 style='margin-left:25%;margin-bottom:25px'>Add Delivery Trucks</h2>")

Button(button_style='primary', description='Add Vehicle', style=ButtonStyle())

Output()

In [25]:
def add_depot(x):
    global depot_results
    location=[d for d in depot_results if select2.value in d['display_name']]
    global depots
    depots=depots.append({'Address':location[0]['display_name'],'Longitude':location[0]['lon'],'Latitude':location[0]['lat'],'Diesel Delivery Amount':-1*max(vehicles['Diesel Capacity']),'Dyed Diesel Delivery Amount':-1*max(vehicles['Starting Dyed Diesel Capacity']),'Delivery Window Start':min(locations['Delivery Window Start']),'Delivery Window End':max(locations['Delivery Window End'])},ignore_index=True)
    return 

display(ipw.HTML("<h2 style='margin-left:25%;margin-bottom:25px'>Add Fuel Depot Locations</h2>"))
depots=pd.DataFrame(columns=['Address','Longitude','Latitude','Diesel Delivery Amount','Dyed Diesel Delivery Amount','Delivery Window Start','Delivery Window End'])
depot_results=my_interact_manual(get_depot_coordinates,address=ipw.Text(value="",placeholder='Type address',description="Location Address",layout=Layout(width='300px'),style={'description_width': 'initial'}),API_key=fixed(key))
run_add_depot=ipw.Button(description='Add Depot',layout=Layout(margin='0 0 15px 0'),button_style='primary')
run_add_depot.on_click(add_depot)

output2=ipw.Output()
def handle_submit2(sender):
    with output2:
        clear_output()
        display(ipw.HTML("<h2>Fuel Depots</h2>"))
        display(depots.drop(columns=['Diesel Delivery Amount','Dyed Diesel Delivery Amount']))  
run_add_depot.on_click(handle_submit2)



display(run_add_depot)

display(output2)

HTML(value="<h2 style='margin-left:25%;margin-bottom:25px'>Add Fuel Depot Locations</h2>")

interactive(children=(Text(value='', description='Location Address', layout=Layout(width='300px'), placeholder…

Button(button_style='primary', description='Add Depot', layout=Layout(margin='0 0 15px 0'), style=ButtonStyle(…

Output()

In [27]:
display(ipw.HTML("<h2 style='margin-left:25%;margin-bottom:25px'>Work Day Start</h2>"))
day_start_time=ipydatetime.TimePicker(description='Day Start Time',style={'description_width': 'initial'},value=datetime.time(7, 00))
display(day_start_time)

HTML(value="<h2 style='margin-left:25%;margin-bottom:25px'>Work Day Start</h2>")

TimePicker(value=datetime.time(7, 0), description='Day Start Time', step=60.0, style=DescriptionStyle(descript…

In [32]:
def get_travel_matrix(all_locations):
    API_key=key.value
    coordinates=all_locations['Longitude'].map(str)+','+all_locations['Latitude'].map(str)
    request = 'https://us1.locationiq.com/v1/matrix/driving/'
    request = request + ';'.join(coordinates) + '?key=' + API_key +'&annotations=distance,duration'
    jsonResult = urllib.request.urlopen(request).read()
    response = json.loads(jsonResult)
    return {'distance_matrix':response['distances'],'duration_matrix':response['durations']}


def create_data_model():
    """Stores the data for the problem."""
    global locations
    global matrix_response
    global vehicles
    global depots
    dup_depots=pd.concat([depots]*len(vehicles),ignore_index=True)
    global all_locations
    start_locations=pd.DataFrame(columns=['Address','Longitude','Latitude','Diesel Delivery Amount','Dyed Diesel Delivery Amount','Delivery Window Start','Delivery Window End'])
    start_locations=start_locations.append({'Address':'1708, Union Landing Road, Cinnaminson Industrial Park, East Riverton, Riverton, Cinnaminson Township, Burlington County, New Jersey, 08077, USA','Longitude':'-74.97965744509614','Latitude':'40.01048502418648','Diesel Delivery Amount':0,'Dyed Diesel Delivery Amount':0,'Delivery Window Start':min(locations['Delivery Window Start']),'Delivery Window End':max(locations['Delivery Window End'])},ignore_index=True)
    for i in range(0,len(vehicles)):
        diesel=vehicles['Diesel Capacity'][i]-vehicles['Starting Diesel Capacity'][i]
        dyed_diesel=vehicles['Dyed Diesel Capacity'][i]-vehicles['Starting Dyed Diesel Capacity'][i]
        start_locations=start_locations.append({'Address':'1708, Union Landing Road, Cinnaminson Industrial Park, East Riverton, Riverton, Cinnaminson Township, Burlington County, New Jersey, 08077, USA','Longitude':'-74.97965744509614','Latitude':'40.01048502418648','Diesel Delivery Amount':diesel,'Dyed Diesel Delivery Amount':dyed_diesel,'Delivery Window Start':min(locations['Delivery Window Start']),'Delivery Window End':max(locations['Delivery Window End'])},ignore_index=True)
    all_locations=pd.concat([start_locations,dup_depots,locations],axis=0).reset_index()
    matrix_response=get_travel_matrix(all_locations)
    day_start=60*(day_start_time.value.hour*60+day_start_time.value.minute)
    time_windows=[]
    for i in range(0,len(all_locations)):
      start=60*(all_locations['Delivery Window Start'][i].hour*60+all_locations['Delivery Window Start'][i].minute)
      end=60*(all_locations['Delivery Window End'][i].hour*60+all_locations['Delivery Window End'][i].minute)
      time_windows.append([start-day_start,end-day_start])
    
    data = {}
    data['distance_matrix'] = matrix_response['distance_matrix']
    data['time_matrix'] = matrix_response['duration_matrix']
    data['time_windows'] = time_windows
    data['num_locations'] = len(all_locations)
    data['demands1'] = all_locations['Diesel Delivery Amount'].tolist()
    data['demands2'] = all_locations['Dyed Diesel Delivery Amount'].tolist()
    data['vehicle_capacities1'] = vehicles['Diesel Capacity'].tolist()
    data['vehicle_capacities2'] = vehicles['Dyed Diesel Capacity'].tolist()
    data['vehicle_max_time']=28800
    data['num_vehicles'] = len(vehicles)
    data['time_per_demand_unit']=3.6
    data['starts'] = [*range(1,len(vehicles)+1)]
    data['ends']=[0]*len(vehicles)
    data['depots']=[*range(len(vehicles)+1,len(vehicles)+len(dup_depots)+1)]
    return data

def get_route(route):
    API_key=key.value
    request = 'https://us1.locationiq.com/v1/directions/driving/'
    request = request + route + '?key=' + API_key +'&overview=full&geometries=geojson'
    jsonResult = urllib.request.urlopen(request).read()
    route_response = json.loads(jsonResult)['routes'][0]['geometry']['coordinates']
    return route_response

def get_map(markers,steps):
    m = folium.Map(location=[40.01048502418648,-74.97965744509614],
      zoom_start=15,tiles='cartodbpositron')
    
    colors=['black', 'blue', 'cadetblue', 'darkblue', 'darkpurple', 'gray', 'lightblue', 'lightgray', 'orange', 'pink', 'purple']
    depot_coords=[]
    for i in range(0,len(depots)):
        depot_coords.append([depots['Latitude'][i],depots['Longitude'][i]])
    truck=1
    for step in steps:
        step=json.loads(step)
        for i in range(0,len(step)):
            step[i]=step[i][::-1]
        folium.PolyLine(step,
                color=colors[(truck-1)%len(colors)],
                weight=5,
                opacity=0.5,
                tooltip='Truck '+str(truck)).add_to(m)
        truck+=1
    truck=1
    for marker in markers:
        pairs=marker.split("|")
        color=colors[(truck-1)%len(colors)]
        stop=1
        for i in range(1,len(pairs)):
            coords=[float(pairs[i].split(",")[0]),float(pairs[i].split(",")[1])]
            popup="Truck "+str(truck)+" Stop "+str(stop)
            if coords in depot_coords:
                folium.Marker(coords,icon=folium.Icon(color='green',icon='oil-can',prefix='fa'),tooltip='Fuel Depot').add_to(m)
            else:
                folium.Marker(coords,icon=folium.Icon(color=color,icon='map-marker',prefix='fa'),tooltip=popup).add_to(m)
            stop+=1
            
        truck+=1
        
    folium.Marker([40.01048502418648,-74.97965744509614],icon=folium.Icon(color='red',icon='home',prefix='fa'),tooltip='7 Oil').add_to(m)
        
    return m

def print_solution(data, manager, routing, solution):
    """Prints solution on console."""
    #print(f'Objective: {solution.ObjectiveValue()}')
    total_distance = 0
    total_load = 0
    map_markers=[]
    route_steps=[]
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        plan_output = '\033[1mRoute for Truck {}:\033[0m\n\n'.format(vehicles['Truck No.'][vehicle_id])
        route_distance = 0
        total_time = 0
        route_load1 = 0
        route_load2 = 0 
        route_coordinates = []
        map_coordinates = []
        time_dimension = routing.GetDimensionOrDie('Time')
        start_load1=vehicles['Diesel Capacity'][vehicle_id]-vehicles['Starting Diesel Capacity'][vehicle_id]
        start_load2=vehicles['Dyed Diesel Capacity'][vehicle_id]-vehicles['Starting Dyed Diesel Capacity'][vehicle_id]
        while not routing.IsEnd(index):
            time_var = time_dimension.CumulVar(index)
            node_index = manager.IndexToNode(index)
            if node_index in data['depots']:
                plan_output += '\t\033[1mLocation Address:\033[0m {0}\n\n\tTruck Refill\n\n'.format(all_locations['Address'][node_index])
            else:
                route_load1 += data['demands1'][node_index]
                route_load2 += data['demands2'][node_index]
                plan_output += '\t\033[1mLocation Address:\033[0m {0}\n\n\t\033[1mCumulative Diesel Delivered:\033[0m {1} gallons\n\t\033[1mCumulative Dyed Diesel Delivered:\033[0m {2} gallons \n\n'.format(all_locations['Address'][node_index], route_load1-start_load1, route_load2-start_load2)
            #plan_output += '\033[1mCumulative Time:\033[0m {0} min\n\n\tTruck Refill\n\n'.format(round(solution.Mean(time_var)/60,0))
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            route_distance += routing.GetArcCostForVehicle(
                previous_index, index, vehicle_id)
            route_coordinates.append(all_locations['Longitude'][node_index]+','+all_locations['Latitude'][node_index])
            map_coordinates.append(all_locations['Latitude'][node_index]+','+all_locations['Longitude'][node_index])
        plan_output += '\t\033[1mReturn To:\033[0m {0}\n\n\033[1mTotal Diesel Delivered:\033[0m {1} gallons\n\033[1mTotal Dyed Diesel Delivered:\033[0m {2} gallons\n'.format(all_locations['Address'][0],
                                                 route_load1-start_load1, route_load2-start_load2)
        plan_output += '\t\033[1mTotal Fuel Delivered:\033[0m {} gallons\n'.format(route_load1 + route_load2 - start_load1 - start_load2)
        plan_output += '\t\033[1mTotal Distance of the route:\033[0m {} miles\n'.format(round(route_distance*0.000621,2))
        print(plan_output)
        
        time_var = time_dimension.CumulVar(index)
        total_distance += route_distance
        total_load += route_load1 + route_load2 - start_load1 - start_load2
        total_time += solution.Min(time_var)
        route_steps.append(str(get_route(';'.join(route_coordinates)+';'+all_locations['Longitude'][0]+','+all_locations['Latitude'][0])))
        time.sleep(0.5)
        map_markers.append('|'.join(map_coordinates))
        
    print('\033[1mTotal distance of all routes:\033[0m {} miles'.format(round(total_distance*0.000621,2)))
    print('\033[1mTotal load of all routes:\033[0m {} gallons'.format(total_load))
    #print('\033[1mTotal Time of all routes:\033[0m {} min'.format(total_time))
    map=get_map(map_markers,route_steps)
    display(map)

def create_demand_evaluator1(data):
    """Creates callback to get demands at each location."""
    _demands = data['demands1']

    def demand_evaluator(manager, from_node):
        """Returns the demand of the current node"""
        return _demands[manager.IndexToNode(from_node)]

    return demand_evaluator


def add_capacity_constraints1(routing, manager, data, demand_evaluator_index):
    """Adds capacity constraint"""
    vehicle_capacity = data['vehicle_capacities1']
    capacity = 'Capacity'
    routing.AddDimensionWithVehicleCapacity(
        demand_evaluator_index,
        10000,
        vehicle_capacity,
        True,  # start cumul to zero
        'Diesel Capacity')

    # Add Slack for reseting to zero unload depot nodes.
    # e.g. vehicle with load 10/15 arrives at node 1 (depot unload)
    # so we have CumulVar = 10(current load) + -15(unload) + 5(slack) = 0.
    capacity_dimension1 = routing.GetDimensionOrDie('Diesel Capacity')
    # Allow to drop reloading nodes with zero cost.
    for node in data['depots']:
        node_index = manager.NodeToIndex(node)
        routing.AddDisjunction([node_index], 0)

    # Allow to drop regular node with a cost.
    for node in range(max(data['depots'])+1, len(data['demands1'])):
        node_index = manager.NodeToIndex(node)
        capacity_dimension1.SlackVar(node_index).SetValue(0)
        routing.AddDisjunction([node_index], 100000)
        
def create_demand_evaluator2(data):
    """Creates callback to get demands at each location."""
    _demands = data['demands2']

    def demand_evaluator(manager, from_node):
        """Returns the demand of the current node"""
        return _demands[manager.IndexToNode(from_node)]

    return demand_evaluator


def add_capacity_constraints2(routing, manager, data, demand_evaluator_index):
    """Adds capacity constraint"""
    vehicle_capacity = data['vehicle_capacities2']
    capacity = 'Capacity'
    routing.AddDimensionWithVehicleCapacity(
        demand_evaluator_index,
        10000,
        vehicle_capacity,
        True,  # start cumul to zero
        'Dyed Diesel Capacity')

    # Add Slack for reseting to zero unload depot nodes.
    # e.g. vehicle with load 10/15 arrives at node 1 (depot unload)
    # so we have CumulVar = 10(current load) + -15(unload) + 5(slack) = 0.
    capacity_dimension2 = routing.GetDimensionOrDie('Dyed Diesel Capacity')
    # Allow to drop reloading nodes with zero cost.
    for node in data['depots']:
        node_index = manager.NodeToIndex(node)
        routing.AddDisjunction([node_index], 0)

    # Allow to drop regular node with a cost.
    for node in range(max(data['depots'])+1, len(data['demands2'])):
        node_index = manager.NodeToIndex(node)
        capacity_dimension2.SlackVar(node_index).SetValue(0)
        routing.AddDisjunction([node_index], 10000000)

def create_time_evaluator(data):
    """Creates callback to get total times between locations."""

    def service_time1(data, node):
        """Gets the service time for the specified location."""
        if node in data['starts']:
            return 0
        else:
            return abs(data['demands1'][node]) * data['time_per_demand_unit']
    
    def service_time2(data, node):
        """Gets the service time for the specified location."""
        if node in data['starts']:
            return 0
        else:
            return abs(data['demands2'][node]) * data['time_per_demand_unit']

    def travel_time(data, from_node, to_node):
        """Gets the travel times between two locations."""
        if from_node == to_node:
            travel_time = 0
        else:
            travel_time = data['time_matrix'][from_node][to_node]
        return travel_time

    _total_time = {}
    # precompute total time to have time callback in O(1)
    for from_node in range(data['num_locations']):
        _total_time[from_node] = {}
        for to_node in range(data['num_locations']):
            if from_node == to_node:
                _total_time[from_node][to_node] = 0
            else:
                _total_time[from_node][to_node] = int(
                    service_time1(data, from_node) + service_time2(data, from_node) + travel_time(
                        data, from_node, to_node))

    def time_evaluator(manager, from_node, to_node):
        """Returns the total time between the two nodes"""
        return _total_time[manager.IndexToNode(from_node)][manager.IndexToNode(
            to_node)]

    return time_evaluator


def add_time_window_constraints(routing, manager, data, time_evaluator):
    """Add Time windows constraint"""
    time = 'Time'
    max_time = data['vehicle_max_time']
    routing.AddDimension(
        time_evaluator,
        0,  # allow waiting time
        max_time,  # maximum time per vehicle
        False,  # don't force start cumul to zero since we are giving TW to start nodes
        time)
    time_dimension = routing.GetDimensionOrDie(time)
    # Add time window constraints for each location except depot
    # and 'copy' the slack var in the solution object (aka Assignment) to print it
    for location_idx, time_window in enumerate(data['time_windows']):
        if location_idx == 0:
            continue
        index = manager.NodeToIndex(location_idx)
        time_dimension.CumulVar(index).SetRange(time_window[0], time_window[1])
        routing.AddToAssignment(time_dimension.SlackVar(index))
    # Add time window constraints for each vehicle start node
    # and 'copy' the slack var in the solution object (aka Assignment) to print it
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        time_dimension.CumulVar(index).SetRange(data['time_windows'][0][0],
                                                data['time_windows'][0][1])
        routing.AddToAssignment(time_dimension.SlackVar(index))
        # Warning: Slack var is not defined for vehicle's end node
        #routing.AddToAssignment(time_dimension.SlackVar(self.routing.End(vehicle_id)))

def main(x):
    """Solve the CVRP problem."""
    # Instantiate the data problem.
    data = create_data_model()

    # Create the routing index manager.
    manager = pywrapcp.RoutingIndexManager(len(data['distance_matrix']),
                                           data['num_vehicles'], data['starts'],data['ends'])

    # Create Routing Model.
    routing = pywrapcp.RoutingModel(manager)


    # Create and register a transit callback.
    def distance_callback(from_index, to_index):
        """Returns the distance between the two nodes."""
        # Convert from routing variable Index to distance matrix NodeIndex.
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        if from_node == to_node:
            distance = 0
        elif from_node in data['depots'] and to_node in data['depots']:
            distance = 1000000#data['vehicle_max_distance']
        else:
            distance = data['distance_matrix'][from_node][to_node]
        return distance

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)

    # Define cost of each arc.
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)


    # Add Capacity constraint.
    demand_evaluator_index1 = routing.RegisterUnaryTransitCallback(
        partial(create_demand_evaluator1(data), manager))
    add_capacity_constraints1(routing, manager, data, demand_evaluator_index1)

    demand_evaluator_index2 = routing.RegisterUnaryTransitCallback(
        partial(create_demand_evaluator2(data), manager))
    add_capacity_constraints2(routing, manager, data, demand_evaluator_index2)
    
     # Add Time Window constraint
    time_evaluator_index = routing.RegisterTransitCallback(
        partial(create_time_evaluator(data), manager))
    add_time_window_constraints(routing, manager, data, time_evaluator_index)
    
    # Setting first solution heuristic.
    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.AUTOMATIC)
    search_parameters.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH)
    search_parameters.time_limit.FromSeconds(1)

    # Solve the problem.
    solution = routing.SolveWithParameters(search_parameters)

    
    # Print solution on console.
    if solution:
        with output4:
          clear_output()
          display(ipw.HTML("<h3>Truck Routes</h3>"))
          print_solution(data, manager, routing, solution)
    else:
        with output4:
          clear_output()
          print('No Solution Found')

display(ipw.HTML("<h2 style='margin-left:25%;margin-bottom:25px'>Build Delivery Routes</h2>"))
solution=ipw.Button(description='Run Optimization',button_style='primary')
solution.on_click(main)
display(solution)

output4=ipw.Output()
display(output4)

HTML(value='<h1>Build Delivery Routes</h1>')

Button(button_style='primary', description='Run Optimization', style=ButtonStyle())

Output()